In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Input Data

In [ ]:
train_data = pd.read_csv('/kaggle/input/c/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/c/titanic/test.csv')

train_data.shape, test_data.shape

# Cek Informasi Data

In [ ]:
print ("\ninfo train")
train_data.info()
print ("\ninfo test")
test_data.info()
print ("\ndescribe train")
train_data.describe()
print ("\ntotal Nan")
train_data.isnull().sum()


# Feature Engineering 

Tangani missing values pada data

pisahkan title (Mr, Mrs, Miss, dst)

pclass_sq = pclass ** 2
age_sq = age ** 2
familySize = sibsp + parch

perhatikan bahwa fare memiliki histogram skewed-right sehingga kita harus menggunakan log transformation

ubah sex menjadi 2 value (0 = female, 1 = male)

boolean check:
hasCabin (check apakah null or not)
isAlone (familysize == 1)




In [ ]:
#check null value
train_data.head()

null value ada di age, cabin, sama embarked

null value di age kita ganti pake median
null value cabin ga perlu kita ubah, karena udh ada feature hasCabin
null value embarked pake modus

In [ ]:
#kita cek dulu titlenya apa aja
titles = train_data['Name'].str.split(',').str[1].str.strip().str.split('.').str[0]

print(titles.value_counts())

In [ ]:
# membuat variabel id untuk output
passengerID = test_data['PassengerId']

# menggabungkan kedua jenis data (train dan test) tanpa jenis 'survived(output yang diminta)', ignore index berfungsi untuk membuat indeks baru 
allData = pd.concat([train_data.drop(['Survived'],axis=1), test_data], ignore_index = True)

# mengisi nilai kosong dengan mengkondisikan kasus setiap variabel
allData['Age'] = allData['Age'].fillna(allData['Age'].median()) # mengambil nilai rata-rata
allData['Fare'] = allData['Fare'].fillna(allData['Fare'].median())
allData['Embarked'] = allData['Embarked'].fillna(allData['Embarked'].mode()[0])

#Feature Engineering Start
allData['Title'] = allData['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

#untuk title yang langka, kita ganti ke rare
allData['Title'] = allData['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

#Replace nama data asing menjadi lebih umum 
allData['Title'] = allData['Title'].replace('Mlle', 'Miss')
allData['Title'] = allData['Title'].replace('Ms', 'Miss')
allData['Title'] = allData['Title'].replace('Mme', 'Mrs')

# membuat hubungan non-linear dari setiap kasus
allData['Pclass_sq'] = allData['Pclass'] ** 2
allData['Age_sq'] = allData['Age'] ** 2
allData['familySize'] = allData['SibSp'] + allData['Parch'] + 1 #tambah 1 utk diri sendiri
allData['isAlone'] = (allData['familySize']==1).astype(int) # Ambil data family == 1 untuk isAlone
allData['HasCabin'] = (allData['Cabin'].notna()).astype(int) # ubah data adanya cabin menjadi bool

#ubah female jadi 0, male jadi 1
sexMap = {'female': 0, 'male': 1}
allData['Sex'] = allData['Sex'].map(sexMap)

#fare harus diubah menggunakan log transform karena histogramnya berbentuk skewed-left
allData['FareLog'] = np.log1p(allData['Fare'])

#pisahin prefix(kata depan ) dari tiket
def get_ticketPrefix(ticket):
    ticket = re.sub(r'[\./]', '', ticket) 
    parts = ticket.split(' ')
    if len(parts) > 1 and not parts[0].isdigit():
        return parts[0]
    else:
        return 'NUM'
allData['ticketPrefix'] = allData['Ticket'].apply(get_ticketPrefix)

#drop kolom yg udh ga kepake
dropColumns = ['Name', 'SibSp', 'Parch', 'Cabin', 'Fare', 'Ticket']
allData = allData.drop(columns=dropColumns)

#pisahin data jadi dua lagi
trainTemp = allData.iloc[:len(train_data)]
testTemp = allData.iloc[len(train_data):]

trainTemp.columns


In [ ]:
# data yang akan diubah menjadi numerikal (bool)
catCol = ['Embarked', 'Title', 'ticketPrefix']

# gabungkan kedua data untuk diubah menjadi bool
allData_encoded = pd.concat([trainTemp, testTemp], ignore_index=True)

# ubah data yang semula hanya satu kolom ( tiket -> tiket a, tiket b, ...) agar bisa diklasifikasikan bool
allData_encoded = pd.get_dummies(allData_encoded, columns=catCol, drop_first=True)

# split kedua data lagi 
train_processed = allData_encoded.iloc[:len(trainTemp)]
test_processed = allData_encoded.iloc[len(trainTemp):]

# Train Nilai 

In [ ]:
from sklearn.model_selection import train_test_split

X = train_processed.drop(['PassengerId'], axis=1) 
y = train_data['Survived'] 

X_test_final = test_processed.drop(['PassengerId'], axis=1) 

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Best Estimators dan Modelling

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

# Inisialisasi model dasar
rf = RandomForestClassifier(random_state=42)

# Distribusi parameter untuk RandomizedSearchCV
param_dist = {
    'n_estimators': randint(100, 1000),
    'max_depth': [4, 6, 8, 10, None],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5)
}

# Randomized search setup
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=25,              # jumlah percobaan acak, boleh kamu naikkan
    cv=5,
    scoring='f1',     # bisa diganti 'f1' untuk dataset imbalanced
    n_jobs=-1,
    random_state=42,
    verbose=2
)

# Fit random search
random_search.fit(X_train_split, y_train_split)

# Cetak hasil terbaik
print("Best Parameters:", random_search.best_params_)
print("Best Cross-Validation Score:", random_search.best_score_)

# Simpan model terbaik
best_model = random_search.best_estimator_

# Latih model
best_model.fit(X_train_split, y_train_split)

# Prediksi Probabilitas

In [ ]:
from sklearn.metrics import log_loss

# Prediksi probabilitas (bukan kelas)
y_proba_val = best_model.predict_proba(X_val_split)
y_proba_positive = y_proba_val[:, 1]

# Hitung log loss
logloss_value = log_loss(y_val_split, y_proba_positive)

print("Log Loss (Validation):", logloss_value)

# Best Treshold

In [ ]:
from sklearn.metrics import accuracy_score

thresholds = np.arange(0.1, 0.9, 0.01)
best_threshold = 0
best_acc = 0

for t in thresholds:
    preds = (y_proba_val[:, 1] >= t).astype(int)
    acc = accuracy_score(y_val_split, preds)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"Best Threshold: {best_threshold:.2f} dengan Akurasi: {best_acc:.4f}")

# Output

In [ ]:
final_proba_test = best_model.predict_proba(X_test_final)[:, 1]

final_predictions = (final_proba_test >= best_threshold).astype(int)

passengerId = test_processed['PassengerId'] 

submission = pd.DataFrame({
    'PassengerId': passengerId, 
    'Survived': final_predictions 
})

submission.to_csv('submission.csv', index=False)
print("Success")
